In [182]:
import numpy as np
from pathlib import Path
import pandas as pd
from mordred import Calculator, descriptors
from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Geometry import Point3D
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate, RepeatedKFold
from pygad import GA
import matplotlib.pyplot as plt
import re

In [ ]:
import sys

#仓库根：从当前目录往上找带 config.py 的目录（git clone 和直接下载 zip 都能用）
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config

In [ ]:
SEED = 114514
HARTREE = 2625.5
BEGIN = 1000
END = 1435

In [184]:
def xyz_to_mol(smiles, path):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))      # 拓扑 + 氢（顺序：重原子，然后氢）
    lines = open(path).read().splitlines()            # 读整个文件
    n = int(lines[0])                                 # 第1行 = 原子数
    xyz = np.array([l.split()[1 : ] for l in lines[2 : 2 + n]], float)  # 取后3列 = 坐标
    conf = Chem.Conformer(n)                          # 建一个空构象
    for i, p in enumerate(xyz):
        conf.SetAtomPosition(i, Point3D(*p))          # 第 i 个原子 ← 第 i 行坐标
    mol.AddConformer(conf, assignId = True)           # 装进分子
    return mol

def read_energy(path):
    text = path.read_text(encoding = "utf-8")
    return float(re.search(r'"total energy"\s*:\s*(-?[\d.eE+-]+)', text).group(1))

def bde_of_bond(folder):
    Ep = read_energy(folder / "parent.json")

    E_v = {}
    for name in ("fragA", "fragB"):                 # 垂直：片段保持父几何
        E_v[name] = read_energy(folder / f"{name}_u1.json")
    v = (E_v["fragA"] + E_v["fragB"] - Ep) * HARTREE
    return v

In [185]:
def filter_features(df):
    var_selector = VarianceThreshold(threshold = 0)
    df_var = var_selector.fit_transform(df)
    selected_columns = df.columns[var_selector.get_support()]
    df1 = pd.DataFrame(df_var, columns = selected_columns)

    corr = df1.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k = 1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
    df1 = df1.drop(columns = to_drop)
    return df1

def embed_features(df, y):
    rf = RandomForestRegressor(
        n_estimators = 100,
        max_depth = None,
        random_state = SEED,
        n_jobs = -1
    )
    Importances = rf.fit(df.values, y).feature_importances_
    feature_importances = pd.Series(Importances, index = df.columns).sort_values(ascending = False)
    df = df[feature_importances.index].iloc[:, : 67]
    return df

def make_fitness(x_part, y_part):
    def fitness(ga_instance, solution, solution_idx):
        selected = np.where(solution == 1)[0]
        if len(selected) == 0:
            return -1e6
        rf = RandomForestRegressor(
            n_estimators = 80,
            max_depth = None,
            random_state = SEED
        )
        score = cross_validate(
            rf, x_part[:, selected], y_part,
            cv = RepeatedKFold(n_splits = 4, n_repeats = 3, random_state = SEED),
            scoring = "r2",
            n_jobs = -1
        )
        return score["test_score"].mean()
    return fitness

In [186]:
# 处理 .xlsx 数据，生成 mol 类
file = config.DATA_TDT / "fixed.xlsx"
df = pd.read_excel(file)[BEGIN : END].reset_index(drop = True)

STORE = config.PARENTS
mols = []
for i in range(len(df)):
    mols.append(xyz_to_mol(df.loc[i, "SMILES"], STORE / f"parent_{i + BEGIN:04d}" / "conf.xyz"))
df["ROMol"] = mols
PandasTools.RenderImagesInAllDataFrames()

In [187]:
TAR = config.PARENTS

PATS = Chem.MolFromSmarts("[#6]-[N+](=O)[O-]")

trigger_wbo = []
max_wbo = []

for cur, i in enumerate(sorted(TAR.glob("parent_*"))[BEGIN : END]):
    mol = Chem.AddHs(df["ROMol"][cur])
    wbo = pd.read_csv(i / "wbo", sep = r"\s+", names=["a", "b", "wbo"])
    wbo_d = {(min(a, b) - 1, max(a, b) - 1) : w for a, b, w in wbo.values}

    all_bond = []
    for b in mol.GetBonds():
        if b.GetBeginAtom().GetAtomicNum() < 2 or b.GetEndAtom().GetAtomicNum() < 2:
            continue
        min_index = min(b.GetBeginAtomIdx(), b.GetEndAtomIdx())
        max_index = max(b.GetBeginAtomIdx(), b.GetEndAtomIdx())
        w = wbo_d.get((min_index, max_index))
        if w is not None:
            all_bond.append(w)
    max_wbo.append(max(all_bond))

print(len(max_wbo))

426


In [188]:
WORK = config.XTB_WORK / "BDE"

BDE_v = []

for i in df.index:
    sub = WORK / f"BDE_{i + BEGIN:04d}"
    sub.mkdir(exist_ok = True)

    vals = []
    for j, name in enumerate(["C-NO2", "N-NO2", "O-NO2"]):
        t_idx = i * 3 + j
        if not (sub / f"{name}").exists():
            continue
        v = bde_of_bond(sub / f"{name}")
        if v is not None:
            vals.append(v)
    
    BDE_v.append(min(vals) if vals else np.nan)
    pd.DataFrame({"BDE_v": BDE_v}).to_csv(WORK / "progress.csv", index = False)

print(len(BDE_v), pd.Series(BDE_v).isna().sum())

426 47


In [189]:
# 计算 Mordred 全量特征
calc = Calculator(descriptors, ignore_3D = True)
df1 = calc.pandas(df["ROMol"])
df1 = df1.apply(pd.to_numeric, errors = "coerce")
df1 = df1.dropna(axis = 1, thresh = int(0.7 * len(df1)))
df1 = df1.fillna(0)
df1.shape

100%|██████████| 426/426 [00:05<00:00, 84.19it/s] 


(426, 1449)

In [190]:
df1 = filter_features(df1)
df1 = embed_features(df1, df["Td(°C)"])

df1["f_BDE"] = BDE_v
df1["has_BDE"] = df1["f_BDE"].notna().astype(int)
df1["f_BDE"] = df1["f_BDE"].fillna(df1["f_BDE"].median())
df1["max_wbo"] = max_wbo
df1.shape

(426, 70)

In [191]:
# 设置初始种群
feats_index = [67, 68, 69]
initial_pop = np.random.randint(0, 2, size=(140, 70))
initial_pop[ : 120, feats_index] = 1
initial_pop

array([[1, 0, 0, ..., 1, 1, 1],
       [1, 0, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       ...,
       [0, 1, 0, ..., 0, 0, 1],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 1, 0, ..., 1, 0, 1]], shape=(140, 70), dtype=int32)

In [ ]:
x_all = df1.values
y_all = df["Td(°C)"]

avg_list = []
def on_gen(ga):
    fit = ga.last_generation_fitness
    if fit is not None:
        avg_list.append(np.mean(fit))

ga = GA(
    sol_per_pop = 140,
    num_genes = 70,
    gene_space = [0, 1],
    gene_type = int,
    fitness_func = make_fitness(x_all, y_all),
    keep_elitism = 2,
    parent_selection_type = "tournament",
    K_tournament = 4,
    num_parents_mating = 15,
    crossover_type = "two_points",
    mutation_percent_genes = 3,
    num_generations = 50,
    on_generation = on_gen,
    parallel_processing = 4
)
ga.run()

ga.plot_fitness()
plt.plot(avg_list)
plt.xlabel("Generation")
plt.ylabel("Average Fitness")
plt.show()

best_solution = ga.best_solution()[0]
res = df1.iloc[:, np.where(best_solution == 1)[0]].columns
res